# Appendix Table: Qualitative Analysis of Predictions

Shows randomly selected passages with GT triples paired with the most similar
predicted triple (by BERTScore). Uses `df_metrics_additional_triple_stats` from
the evaluation pkl.

The output is a starting point — passages may be cherry-picked for the final paper.

In [1]:
import os
import sys
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), 'src'))
sys.path.insert(0, os.path.join(os.path.abspath('../../..'), 'src'))

from stats.evaluation.load_results import load_from_wiki_eval_result

In [2]:
# --- CONFIGURE THIS ---
# Scoring modes:
# - Legacy (reproduces ICML submission): use *_legacy_with_kg pkl
# - Fixed (post-rebuttal, score_empty_predictions_as_zero=true): use *_fixed_with_kg pkl
# See src/evaluation/README.md for details on the scoring bug.

# --- Fixed scoring + KG snapshots:
PKL_PATH = '/path/to/storage/emerge/output/experiments/s0x_evaluate_predictions/20260324_submitted_icml_fixed_with_kg/wiki_eval_result.pkl'
# --- Legacy scoring + KG snapshots (reproduces paper values):
# PKL_PATH = '/path/to/storage/emerge/output/experiments/s0x_evaluate_predictions/20260324_submitted_icml_legacy_with_kg/wiki_eval_result.pkl'
# --- Fixed scoring, no KG (relik-cie x-triples missing):
# PKL_PATH = '/path/to/storage/emerge/output/experiments/s0x_evaluate_predictions/20260217_submitted_icml_fixed/wiki_eval_result.pkl'

# Model to show predictions for
MODEL = 'edc-plus-open-ai/gpt-5.1/non-canonicalized'
MODEL_DISPLAY = 'EDC+ GPT-5.1'

# BERTScore evaluator model to use for matching
EVALUATOR_MODEL = 'bert-base'

# Number of candidate passages to display
N_PASSAGES = 5

# Max passage length (words) for readability
MAX_PASSAGE_WORDS = 80

# Random seed for reproducibility
RANDOM_SEED = 42

In [3]:
# Load data
(df_cie, _, df_gt, _, df_inst, df_additional) = load_from_wiki_eval_result(PKL_PATH)

print(f'df_additional.shape: {df_additional.shape}')
print(f'df_inst.shape: {df_inst.shape}')

# Filter to GT-side matches with BERTScore for the target model
df_matches = df_additional[
    (df_additional['metric'] == 'gj-triple-matches-w-gold') &
    (df_additional['evaluator_model'] == EVALUATOR_MODEL) &
    (df_additional['model'] == MODEL)
].copy()

print(f'Matches for {MODEL}: {len(df_matches)}')
print(f'tkgu_types: {df_matches.tkgu_type.unique().tolist()}')

df_additional.shape: (1914306, 19)
df_inst.shape: (3500, 13)


Matches for edc-plus-open-ai/gpt-5.1/non-canonicalized: 22533
tkgu_types: ['x-triples', 'e-triples', 'ee-triples', 'ee-kg-triples', 'd-triples']


In [4]:
# Join passage and snapshot_year
df_matches = df_matches.merge(
    df_inst[['hash_id', 'passage', 'snapshot_year']],
    on='hash_id', how='inner'
)

# Filter to passages with reasonable length
df_matches['n_words'] = df_matches['passage'].str.split().str.len()
df_matches = df_matches[df_matches['n_words'] <= MAX_PASSAGE_WORDS].copy()

print(f'After passage length filter: {len(df_matches)} rows, {df_matches.hash_id.nunique()} unique passages')

After passage length filter: 8361 rows, 1955 unique passages


In [5]:
# Find passages that have multiple TKGU types (more interesting for the table)
tkgu_per_passage = df_matches.groupby('hash_id')['tkgu_type'].nunique()
multi_tkgu_passages = tkgu_per_passage[tkgu_per_passage >= 2].index

df_candidates = df_matches[df_matches['hash_id'].isin(multi_tkgu_passages)].copy()
print(f'Passages with >= 2 TKGU types: {len(multi_tkgu_passages)}')

# Sample N passages
candidate_hash_ids = df_candidates['hash_id'].unique()
rng = np.random.RandomState(RANDOM_SEED)
selected = rng.choice(candidate_hash_ids, size=min(N_PASSAGES, len(candidate_hash_ids)), replace=False)
print(f'Selected {len(selected)} passages')

Passages with >= 2 TKGU types: 1391
Selected 5 passages


In [6]:
# TKGU type display mapping
tkgu_to_latex = {
    'x-triples': r'\opexists',
    'e-triples': r'\opadd',
    'ee-triples': r'\opmintadd',
    'ee-kg-triples': r'\opinfer',
    'd-triples': r'\opdeprecate',
}

tkgu_order = {'x-triples': 0, 'e-triples': 1, 'ee-triples': 2,
              'ee-kg-triples': 3, 'd-triples': 4}

def score_to_color(score):
    """Map BERTScore (0-1) to blue intensity for cellcolor."""
    intensity = int(score * 60)
    return f'\\cellcolor{{blue!{intensity}}}'

def format_triple_latex(head, rel, tail):
    return f'$\\langle${head}; {rel}; {tail}$\\rangle$'

# Generate LaTeX
all_passage_blocks = []

for idx, hid in enumerate(selected):
    passage_rows = df_candidates[df_candidates['hash_id'] == hid].copy()
    passage_text = passage_rows['passage'].iloc[0]
    snapshot = int(passage_rows['snapshot_year'].iloc[0])

    # Sort by tkgu_type then by score descending
    passage_rows['tkgu_rank'] = passage_rows['tkgu_type'].map(tkgu_order)
    passage_rows = passage_rows.sort_values(['tkgu_rank', 'pred_score'], ascending=[True, False])

    # Build rows
    triple_rows = []
    for _, row in passage_rows.iterrows():
        tkgu_latex = tkgu_to_latex.get(row['tkgu_type'], row['tkgu_type'])
        gt_triple = format_triple_latex(
            row['gt_triple_head_label'], row['gt_triple_relation_label'], row['gt_triple_tail_label'])
        pred_triple = format_triple_latex(
            row['pred_triple_head_label'], row['pred_triple_relation_label'], row['pred_triple_tail_label'])
        score = row['pred_score']
        color = score_to_color(score)
        triple_rows.append(
            f'{tkgu_latex} & {gt_triple} & {pred_triple} & {color}{score:.3f} \\\\'
        )

    block = f"""\\toprule
\\multicolumn{{4}}{{p{{0.98\\textwidth}}}}{{\\textbf{{Passage {idx+1}. (KG Snapshot {snapshot})}} {passage_text}}} \\\\
\\midrule
\\textbf{{TKGU}} & \\textbf{{Ground Truth}} & \\textbf{{Predicted}} & \\textbf{{BERTScore}} \\\\
\\midrule
""" + "\n".join(triple_rows) + "\n\\bottomrule"

    all_passage_blocks.append(block)

body = "\n".join(all_passage_blocks)

latex = f"""\\renewcommand{{\\arraystretch}}{{1.12}}
\\begin{{table}}[H]
\\centering
\\small
\\setlength{{\\tabcolsep}}{{4pt}}
\\rowcolors{{3}}{{gray!8}}{{white}}
\\begin{{tabular}}{{
  p{{0.09\\textwidth}}
  p{{0.38\\textwidth}}
  p{{0.37\\textwidth}}
  p{{0.09\\textwidth}}
}}
{body}
\\end{{tabular}}
\\caption{{
Randomly selected examples of {MODEL_DISPLAY} predictions. While most predicted triples are semantically correct, they often differ from the ground truth triples, which depend strongly on the underlying knowledge graph schema. Each ground truth triple is paired with its most similar predicted triple according to BERTScore.
}}
\\label{{table:appendix:prediction:examples}}
\\end{{table}}
\\renewcommand{{\\arraystretch}}{{1}}"""

print(latex)

\renewcommand{\arraystretch}{1.12}
\begin{table}[H]
\centering
\small
\setlength{\tabcolsep}{4pt}
\rowcolors{3}{gray!8}{white}
\begin{tabular}{
  p{0.09\textwidth}
  p{0.38\textwidth}
  p{0.37\textwidth}
  p{0.09\textwidth}
}
\toprule
\multicolumn{4}{p{0.98\textwidth}}{\textbf{Passage 1. (KG Snapshot 2020)} Excerpt from the Wikipedia page describing Abu Mahdi al-Muhandis: On 3 January 2020, al-Muhandis was killed in an airstrike while traveling in a convoy near Baghdad International Airport along with Qasem Soleimani as retaliation for the embassy attack a few days prior.} \\
\midrule
\textbf{TKGU} & \textbf{Ground Truth} & \textbf{Predicted} & \textbf{BERTScore} \\
\midrule
\opmintadd & $\langle$assassination of Qasem Soleimani; location; Baghdad International Airport$\rangle$ & $\langle$Qasem Soleimani; place of death; Baghdad International Airport$\rangle$ & \cellcolor{blue!52}0.867 \\
\opmintadd & $\langle$Baghdad International Airport; significant event; assassination of Qasem Sol